# OpenReview Acceptance Predictor: Colab Pipeline

This notebook is designed to run sequentially from a prebuilt processed dataset. It installs dependencies, restores the committed compressed dataset if needed, creates splits, trains the baselines, builds retrieval, runs RAG, and optionally runs LoRA fine-tuning.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REPO_URL = 'https://github.com/trinayhari/ml-paper-evaluator.git'
CLONE_DIR = '/content/ml-paper-evaluator'
%cd /content
!git clone {REPO_URL} {CLONE_DIR} || true
%cd /content/ml-paper-evaluator


In [ ]:
!python -m pip install -r requirements.txt


## Configure dataset and run options

Set `PROCESSED_DATA` to the extracted dataset you already created locally. `OUTPUT_ROOT` is where all generated splits, models, metrics, and predictions will be written.


In [ ]:
PROCESSED_DATA = '/content/ml-paper-evaluator/data/processed/papers.jsonl.gz'
OUTPUT_ROOT = '/content/drive/MyDrive/openreview/colab_pipeline_run'
SEED = 42
TEST_VENUES = []
RAG_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
RAG_LIMIT = 100
RAG_K = 5
RUN_LORA = False
LORA_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

ROOT = Path('/content/ml-paper-evaluator')
processed_path = Path(PROCESSED_DATA)
output_root = Path(OUTPUT_ROOT)
splits_dir = output_root / 'splits'
baselines_dir = output_root / 'baselines'
retrieval_dir = output_root / 'retrieval'
rag_out = output_root / 'rag_predictions.jsonl'
lora_dir = output_root / 'lora_model'

def run_cmd(cmd):
    print('\n==>', ' '.join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

if not processed_path.exists():
    part_candidates = sorted(processed_path.parent.glob(processed_path.name + '.part-*'))
    if part_candidates:
        run_cmd(['python', 'scripts/00_restore_dataset.py', '--out', str(processed_path)])

assert processed_path.exists(), f'Missing processed dataset: {processed_path}'
print('Using dataset:', processed_path)


## Option A: run the whole pipeline in one command

This cell is the easiest path for someone who just wants to execute everything after setting the dataset path above.


In [ ]:
if output_root.exists():
    shutil.rmtree(output_root)

cmd = [
    'python', 'scripts/08_run_colab_pipeline.py',
    '--input', str(processed_path),
    '--output-root', str(output_root),
    '--seed', str(SEED),
    '--clean-output',
    '--rag-model', RAG_MODEL,
    '--rag-limit', str(RAG_LIMIT),
    '--rag-k', str(RAG_K),
]
if TEST_VENUES:
    cmd.extend(['--test-venues', *TEST_VENUES])
if RUN_LORA:
    cmd.extend(['--run-lora', '--lora-model', LORA_MODEL])
run_cmd(cmd)


## Option B: run each stage explicitly

Use the next cells if you want to see each stage run separately. They use the same dataset and output paths configured above.


In [ ]:
split_cmd = [
    'python', 'scripts/03_make_splits.py',
    '--input', str(processed_path),
    '--out-dir', str(splits_dir),
    '--seed', str(SEED),
]
if TEST_VENUES:
    split_cmd.extend(['--test-venues', *TEST_VENUES])
run_cmd(split_cmd)


In [ ]:
run_cmd([
    'python', 'scripts/04_train_baselines.py',
    '--train', str(splits_dir / 'train.jsonl'),
    '--dev', str(splits_dir / 'dev.jsonl'),
    '--test', str(splits_dir / 'test.jsonl'),
    '--out', str(baselines_dir),
])


In [ ]:
run_cmd([
    'python', 'scripts/05_build_retrieval_index.py',
    '--train', str(splits_dir / 'train.jsonl'),
    '--out', str(retrieval_dir),
])


In [ ]:
run_cmd([
    'python', 'scripts/06_rag_predict.py',
    '--test', str(splits_dir / 'test.jsonl'),
    '--index-dir', str(retrieval_dir),
    '--out', str(rag_out),
    '--model', RAG_MODEL,
    '--limit', str(RAG_LIMIT),
    '--k', str(RAG_K),
])


## Optional LoRA fine-tuning

Run this only if your Colab runtime has a GPU with enough memory and you intentionally want the fine-tuning step.


In [ ]:
if RUN_LORA:
    run_cmd([
        'python', 'scripts/07_finetune_lora.py',
        '--train', str(splits_dir / 'train.jsonl'),
        '--dev', str(splits_dir / 'dev.jsonl'),
        '--model', LORA_MODEL,
        '--out', str(lora_dir),
    ])
else:
    print('Skipping LoRA because RUN_LORA is False')


In [ ]:
summary_path = output_root / 'run_summary.json'
if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    summary
else:
    print('No run_summary.json found. If you ran stages manually, inspect the output folders directly.')
    print('Output root:', output_root)


In [ ]:
metrics_path = baselines_dir / 'metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    metrics
else:
    print('Missing metrics file:', metrics_path)
